In [1]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import log_loss, roc_auc_score

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("[warning] xgboost not installed — skipping XGBoost. "
          "Install with: pip install xgboost")

# ----------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------
TRAIN_PATH = "/Users/ak/Documents/UNSW Classes/DATA5925/CODE/Train and test/train_split.csv"
TEST_PATH  = "/Users/ak/Documents/UNSW Classes/DATA5925/CODE/Train and test/test_split.csv"
OUTPUT_DIR = "/Users/ak/Documents/UNSW Classes/DATA5925/CODE/single_vs_multi_outputs"

TARGET = "pres.abs"
SPECIES_COL = "Species"

# Using easting/northing rather than long/lat, per the team's Week 3 decision.
NUMERIC_FEATURES = [
    "easting", "northing", "disturb", "rainann",
    "soildepth", "soilfert", "tempann", "topo",
]

CLIP = 1e-6  # clip predicted probabilities away from 0/1 before log-loss


# ----------------------------------------------------------------------
# MODEL ZOO
# ----------------------------------------------------------------------
def build_models():
    models = {
        "LogisticRegression": lambda: LogisticRegression(
            max_iter=2000, class_weight="balanced"
        ),
        "RandomForest": lambda: RandomForestClassifier(
            n_estimators=500, max_depth=None, min_samples_leaf=3,
            class_weight="balanced", random_state=42, n_jobs=-1
        ),
        "MLP": lambda: MLPClassifier(
            hidden_layer_sizes=(32, 16), activation="relu", alpha=1e-3,
            learning_rate_init=1e-3, max_iter=2000, random_state=42
        ),
    }
    if HAS_XGB:
        models["XGBoost"] = lambda: XGBClassifier(
            n_estimators=400, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            eval_metric="logloss", random_state=42,
        )
    return models


def clipped_log_loss(y_true, y_pred):
    y_pred = np.clip(y_pred, CLIP, 1 - CLIP)
    return log_loss(y_true, y_pred, labels=[0, 1])


def safe_auc(y_true, y_pred):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, y_pred)


def maybe_set_xgb_weight(model, y_train):
    if HAS_XGB and isinstance(model, XGBClassifier):
        pos = max(y_train.sum(), 1)
        neg = max(len(y_train) - y_train.sum(), 1)
        model.set_params(scale_pos_weight=neg / pos)


# ----------------------------------------------------------------------
# (A) SINGLE-SPECIES
# ----------------------------------------------------------------------
def run_single_species(train, test, model_name, model_fn):
    rows, preds_all = [], np.zeros(len(test))

    for species, train_sp in train.groupby(SPECIES_COL):
        mask = (test[SPECIES_COL] == species).values
        test_sp = test.loc[mask]
        if test_sp.empty:
            continue

        X_train, y_train = train_sp[NUMERIC_FEATURES], train_sp[TARGET].astype(int)
        X_test, y_test = test_sp[NUMERIC_FEATURES], test_sp[TARGET].astype(int)

        model = model_fn()
        maybe_set_xgb_weight(model, y_train)
        pipe = Pipeline([("prep", StandardScaler()), ("clf", model)])
        pipe.fit(X_train, y_train)
        p = pipe.predict_proba(X_test)[:, 1]
        preds_all[mask] = p

        rows.append({
            "species": species, "n_train": len(train_sp), "n_test": len(test_sp),
            "prevalence_train": y_train.mean(),
            "log_loss_single": clipped_log_loss(y_test, p),
            "auc_single": safe_auc(y_test, p),
        })

    df = pd.DataFrame(rows).sort_values("species").reset_index(drop=True)
    overall = clipped_log_loss(test[TARGET].astype(int), preds_all)
    return df, preds_all, overall


# ----------------------------------------------------------------------
# (B) MULTI-SPECIES (additive Species one-hot)
# ----------------------------------------------------------------------
def run_multi_species(train, test, model_name, model_fn, interactions=False):
    """
    interactions=False -> Species is just an additive one-hot feature.
    interactions=True  -> Species one-hot is ALSO multiplied against every
                           numeric feature (Species x easting, etc.), giving
                           the pooled model species-specific slopes.
    """
    X_train_raw = train[NUMERIC_FEATURES + [SPECIES_COL]]
    X_test_raw = test[NUMERIC_FEATURES + [SPECIES_COL]]
    y_train, y_test = train[TARGET].astype(int), test[TARGET].astype(int)

    ohe = OneHotEncoder(handle_unknown="ignore")
    pre_transformers = [
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", ohe, [SPECIES_COL]),
    ]
    if interactions:
        # scikit-learn doesn't have a one-line "cross numeric with dummy"
        # transformer, so build the crossed columns manually and feed them
        # in as extra pre-scaled numeric features.
        species_dummies_train = pd.get_dummies(train[SPECIES_COL], prefix="sp")
        species_dummies_test = pd.get_dummies(test[SPECIES_COL], prefix="sp")
        species_dummies_test = species_dummies_test.reindex(
            columns=species_dummies_train.columns, fill_value=0
        )

        cross_train = pd.DataFrame(index=train.index)
        cross_test = pd.DataFrame(index=test.index)
        for col in NUMERIC_FEATURES:
            for sp_col in species_dummies_train.columns:
                cross_train[f"{col}_x_{sp_col}"] = train[col].values * species_dummies_train[sp_col].values
                cross_test[f"{col}_x_{sp_col}"] = test[col].values * species_dummies_test[sp_col].values

        X_train_full = pd.concat(
            [train[NUMERIC_FEATURES].reset_index(drop=True),
             species_dummies_train.reset_index(drop=True),
             cross_train.reset_index(drop=True)], axis=1
        )
        X_test_full = pd.concat(
            [test[NUMERIC_FEATURES].reset_index(drop=True),
             species_dummies_test.reset_index(drop=True),
             cross_test.reset_index(drop=True)], axis=1
        )
        model = model_fn()
        maybe_set_xgb_weight(model, y_train)
        pipe = Pipeline([("scale", StandardScaler()), ("clf", model)])
        pipe.fit(X_train_full, y_train)
        p = pipe.predict_proba(X_test_full)[:, 1]
    else:
        preprocessor = ColumnTransformer(pre_transformers)
        model = model_fn()
        maybe_set_xgb_weight(model, y_train)
        pipe = Pipeline([("prep", preprocessor), ("clf", model)])
        pipe.fit(X_train_raw, y_train)
        p = pipe.predict_proba(X_test_raw)[:, 1]

    overall = clipped_log_loss(y_test, p)

    suffix = "multi_int" if interactions else "multi"
    rows = []
    for species, test_sp in test.groupby(SPECIES_COL):
        mask = (test[SPECIES_COL] == species).values
        y_sp = test_sp[TARGET].astype(int)
        p_sp = p[mask]
        rows.append({
            "species": species,
            f"log_loss_{suffix}": clipped_log_loss(y_sp, p_sp),
            f"auc_{suffix}": safe_auc(y_sp, p_sp),
        })
    df = pd.DataFrame(rows).sort_values("species").reset_index(drop=True)
    return df, p, overall


# ----------------------------------------------------------------------
# PLOTTING
# ----------------------------------------------------------------------
def plot_model_comparison(merged, model_name, output_dir):
    species = merged["species"]
    x = np.arange(len(species))
    width = 0.25

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.bar(x - width, merged["log_loss_single"], width, label="Single-species")
    ax.bar(x, merged["log_loss_multi"], width, label="Multi-species")
    if "log_loss_multi_int" in merged.columns:
        ax.bar(x + width, merged["log_loss_multi_int"], width, label="Multi + interactions")

    ax.set_xticks(x)
    ax.set_xticklabels(species, rotation=40, ha="right")
    ax.set_ylabel("Log-loss (lower is better)")
    ax.set_title(f"{model_name}: per-species log-loss by approach")
    ax.legend()
    fig.tight_layout()
    path = os.path.join(output_dir, f"{model_name}_comparison.png")
    fig.savefig(path, dpi=150)
    plt.close(fig)
    print(f"[saved] {path}")


def plot_prevalence_vs_gain(all_merged, output_dir):
    """Scatter: does pooling (multi vs single) help low-prevalence species more?"""
    fig, ax = plt.subplots(figsize=(7, 5))
    for model_name, merged in all_merged.items():
        gain = merged["log_loss_single"] - merged["log_loss_multi"]  # positive = multi wins
        ax.scatter(merged["prevalence_train"], gain, label=model_name, alpha=0.8)
    ax.axhline(0, color="grey", linewidth=1, linestyle="--")
    ax.set_xlabel("Training prevalence")
    ax.set_ylabel("Log-loss gain from pooling (single - multi); >0 = multi wins")
    ax.set_title("Does multi-species pooling help rarer species more?")
    ax.legend()
    fig.tight_layout()
    path = os.path.join(output_dir, "prevalence_vs_gain.png")
    fig.savefig(path, dpi=150)
    plt.close(fig)
    print(f"[saved] {path}")


# ----------------------------------------------------------------------
# MAIN
# ----------------------------------------------------------------------
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    train = pd.read_csv(TRAIN_PATH)
    test = pd.read_csv(TEST_PATH)

    print(f"Train: {train.shape}, Test: {test.shape}")
    print(f"Species in train: {train[SPECIES_COL].nunique()}, "
          f"overall prevalence: {train[TARGET].mean():.4f}\n")

    models = build_models()
    summary_rows = []
    all_merged = {}

    for model_name, model_fn in models.items():
        print(f"Running {model_name} ...")

        single_df, _, single_ll = run_single_species(train, test, model_name, model_fn)
        multi_df, _, multi_ll = run_multi_species(train, test, model_name, model_fn, interactions=False)
        multi_int_df, _, multi_int_ll = run_multi_species(train, test, model_name, model_fn, interactions=True)

        merged = single_df.merge(multi_df, on="species").merge(multi_int_df, on="species")
        merged["best_approach"] = merged[
            ["log_loss_single", "log_loss_multi", "log_loss_multi_int"]
        ].idxmin(axis=1).str.replace("log_loss_", "", regex=False)

        all_merged[model_name] = merged

        csv_path = os.path.join(OUTPUT_DIR, f"{model_name}_per_species.csv")
        merged.to_csv(csv_path, index=False)
        print(f"[saved] {csv_path}")

        plot_model_comparison(merged, model_name, OUTPUT_DIR)

        print(f"\n=== {model_name} — overall log-loss ===")
        print(f"  single: {single_ll:.4f} | multi: {multi_ll:.4f} | "
              f"multi+interactions: {multi_int_ll:.4f}")
        print(merged[[
            "species", "n_test", "prevalence_train",
            "log_loss_single", "log_loss_multi", "log_loss_multi_int", "best_approach"
        ]].to_string(index=False))
        print()

        summary_rows.append({
            "model": model_name,
            "overall_log_loss_single": single_ll,
            "overall_log_loss_multi": multi_ll,
            "overall_log_loss_multi_interactions": multi_int_ll,
            "best_overall": min(
                [("single", single_ll), ("multi", multi_ll), ("multi_int", multi_int_ll)],
                key=lambda t: t[1]
            )[0],
        })

    summary_df = pd.DataFrame(summary_rows)
    summary_path = os.path.join(OUTPUT_DIR, "summary_across_models.csv")
    summary_df.to_csv(summary_path, index=False)
    print(f"[saved] {summary_path}")

    plot_prevalence_vs_gain(all_merged, OUTPUT_DIR)

    print("\n\n===== SUMMARY ACROSS MODELS =====")
    print(summary_df.to_string(index=False))

    return all_merged, summary_df


if __name__ == "__main__":
    main()

Train: (4102, 14), Test: (1026, 14)
Species in train: 8, overall prevalence: 0.0631

Running LogisticRegression ...
[saved] /Users/ak/Documents/UNSW Classes/DATA5925/CODE/single_vs_multi_outputs/LogisticRegression_per_species.csv
[saved] /Users/ak/Documents/UNSW Classes/DATA5925/CODE/single_vs_multi_outputs/LogisticRegression_comparison.png

=== LogisticRegression — overall log-loss ===
  single: 0.4456 | multi: 0.5190 | multi+interactions: 0.4531
                   species  n_test  prevalence_train  log_loss_single  log_loss_multi  log_loss_multi_int best_approach
          Cacophis kreftii     128          0.031189         0.352310        0.326458            0.312224     multi_int
   Calyptotis scutirostrum     128          0.077973         0.393658        0.551816            0.464888        single
Coeranoscincus reticulatus     128          0.161793         0.670805        1.113536            1.054516        single
           Egernia mcpheei     128          0.019493         0.42668